**CI twin of `ch20-workflow-model-card.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import train_test_split

num = ["bill_length_mm", "bill_depth_mm",
       "flipper_length_mm", "body_mass_g"]
cat = ["island", "sex"]
df = load_csv("penguins").dropna(subset=num + cat)
y = df["species"]
print(f"{len(df)} birds, three classes:",
      y.value_counts().to_dict())

Xtr, Xte, ytr, yte = train_test_split(
    df[num + cat], y, test_size=0.25, random_state=42, stratify=y)
print(f"training {len(Xtr)}, sealed test {len(Xte)}")

lazy = (yte == ytr.value_counts().idxmax()).mean()
print(f"lazy baseline (always '{ytr.value_counts().idxmax()}'): {lazy:.3f}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier)
from sklearn.model_selection import cross_val_score

prep = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])
candidates = {
    "logistic": LogisticRegression(max_iter=1000),
    "forest":   RandomForestClassifier(n_estimators=100, random_state=0),
    "boosted":  HistGradientBoostingClassifier(random_state=0),
}
for name, model in candidates.items():
    pipe = Pipeline([("prep", prep), ("model", model)])
    s = cross_val_score(pipe, Xtr, ytr, cv=5)
    print(f"{name:9}: CV {s.mean():.3f} ± {s.std():.3f}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)

search = GridSearchCV(
    Pipeline([("prep", prep),
              ("model", LogisticRegression(max_iter=1000))]),
    {"model__C": [0.01, 0.1, 1, 10]}, cv=5)
search.fit(Xtr, ytr)
print(f"CV chose {search.best_params_} at {search.best_score_:.3f}\n")

pred = search.predict(Xte)
print(f"SEALED TEST accuracy: {accuracy_score(yte, pred):.3f}\n")
print(classification_report(yte, pred))
print(confusion_matrix(yte, pred))

In [ ]:
card = f"""
# Model card — Palmer Penguin Species Classifier v1.0

## Model details
Logistic regression (scikit-learn pipeline: standardized numeric
features + one-hot island/sex), C=10 chosen by 5-fold CV.
Trained 2026-07 on 249 birds; evaluated once on 84 held-out birds.

## Intended use
Field identification of Adelie / Chinstrap / Gentoo penguins at Palmer
Archipelago stations, from four body measurements plus island and sex.
Users: station rangers. Decisions supported: routine survey logging —
NOT conservation-critical determinations.

## Training data
Palmer Penguins (Gorman et al., CC0), 2007–2009 field seasons,
333 of 344 birds (rows missing sex were dropped — sex-unknown birds
are therefore outside this model's experience).

## Metrics (held-out test set, n=84)
accuracy 0.988 | per class — Adelie: precision 1.00, recall 0.97;
Chinstrap: precision 0.94, recall 1.00; Gentoo: precision 1.00,
recall 1.00.

## Limitations & failure modes
- The known failure lives at the Adelie–Chinstrap boundary (the one
  test error; both species overlap in bill/flipper space).
- Trained on 2007–2009 Palmer Archipelago birds only.

## When NOT to use
Other regions or penguin species; birds measured with different
protocols; sex-unknown birds; any conservation-critical decision
without a human in the loop.
"""
print(card)

In [ ]:
results = {}
for name, model in [
    ("logistic", LogisticRegression(max_iter=1000)),
    ("forest", RandomForestClassifier(n_estimators=100, random_state=0)),
    ("boosted", HistGradientBoostingClassifier(random_state=0)),
]:
    pipe = Pipeline([("prep", prep), ("model", model)])
    results[name] = round(float(cross_val_score(pipe, Xtr, ytr, cv=5).mean()), 3)

run_tests([
    ("compared on training data only, by CV", results,
     {"logistic": 0.992, "forest": 0.988, "boosted": 0.988}),
])

In [ ]:
def card_metrics(y_true, y_pred, labels):
    out = {}
    for lab in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == lab and p == lab)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != lab and p == lab)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == lab and p != lab)
        precision = round(tp / (tp + fp), 4) if tp + fp else 0.0
        recall = round(tp / (tp + fn), 4) if tp + fn else 0.0
        out[lab] = (precision, recall)
    return out

run_tests([
    ("three-class card block",
     card_metrics(["A", "A", "B", "B", "B", "C"],
                  ["A", "B", "B", "B", "A", "C"],
                  ["A", "B", "C"]),
     {"A": (0.5, 0.5), "B": (0.6667, 0.6667), "C": (1.0, 1.0)}),
    ("a class the model never predicts",
     card_metrics(["A", "B"], ["A", "A"], ["A", "B"]),
     {"A": (0.5, 1.0), "B": (0.0, 0.0)}),
])